# oracle_run — trần của việc CHỌN ĐOẠN, và bộ dữ liệu để thiết kế hàm chọn

**~10 phút GPU.** Bật GPU T4, Internet On, **Save & Run All (Commit)**.

Đã đếm trên CPU 17/08: trong 314 gold của dev300, **31 câu hụt top-5** ở cấu hình chốt
(M=20, K=20, `max`, n=2) — 24 còn trong top-100, 7 nằm ngoài (D chịu).
Trong 24 cái đó: **22 văn bản có >20 đoạn**, và **15 trong số đó đã được chấm sâu**
(CE hạng 1-20) mà vẫn thua → nghi can số một là **chọn nhầm 20 đoạn**.

Notebook này KHÔNG cải tiến gì. Nó chấm CE trên **TOÀN BỘ** đoạn của 24 văn bản gold đó
(~5.100 đoạn) và **lưu điểm TỪNG ĐOẠN**. Hai thứ thu được:

1. **Trần chặt của cả hướng chọn đoạn.** Cho phép chọn đoạn hoàn hảo trong 125 thì cứu
   được mấy câu? Không hàm chọn đoạn nào vượt được con số này.
2. **`oracle_chunks_dev300.json` — tài sản.** Có nó rồi thì mọi hàm chọn đoạn sau này
   (embedding, BM25 mức đoạn, lai) đo được **trên CPU vài giây**, không cần GPU nữa.
   Đúng khuôn đã làm deepchunk rẻ: ghi vào khoá riêng → một lượt GPU, quét mãi về sau.

Chấm luôn ở **hai kích thước đoạn** (hiện tại ~686 ký tự, và gộp lên ~1.800) vì gộp
vào một phiên rẻ hơn chạy hai lượt — mỗi lượt Kaggle tốn ~40 phút setup.

**Đọc kết quả — Bước 5 in thẳng:**

| số câu được kéo lên top-5 | kết luận |
|---|---|
| **≥ 6** | chọn đoạn đúng là nút thắt → viết `pick_chunks` bản embedding, trần ≥ +2,0 |
| **≤ 2** | KHÔNG phải vấn đề chọn đoạn → đừng đốt 2,5h, đổi hướng |

Đây là chặn TRÊN, đọc hơi lạc quan: cải tiến toàn cục thì văn bản đối thủ cũng được nâng
theo (rủi ro "phồng điểm mọi văn bản" ở mục `max` vs `replace` của CLAUDE.md).

## Cài package + trỏ dataset

**Upload trước khi chạy — chỉ MỘT file mới:** `Ketqua_E/scores_dev300_deep_M20_K20.json`.
Mọi thứ khác (`deep_chunk.py`, `rerank.py`, `rerank_from_d.py`,
`make_candidates_fallback.py`, `dev_300_locked.json`, `selected-contexts/`) đã có sẵn
trong dataset từ các lượt trước.

Không cần `bm25_top100_dev300.json` (100MB): thứ tự BM25 suy được từ khoá `bm25` trong
file scores — đã kiểm trên CPU 17/08, dựng lại ra đúng 0.8883 / 0.9083.

In [ ]:
!pip install -q sentence-transformers

import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
INPUT_DIR = "/kaggle/input/project-ir"     # đổi đúng slug — xem output bên dưới
sys.path.append(INPUT_DIR)
!ls /kaggle/input

In [ ]:
import json, time
from pathlib import Path

from rerank import load_reranker
from rerank_from_d import blend_bm25_first
from make_candidates_fallback import chunks_of, read_passage
import deep_chunk as DC

DEV_GOLD = f"{INPUT_DIR}/dev_300_locked.json"
SCORES   = f"{INPUT_DIR}/scores_dev300_deep_M20_K20.json"
CTX_DIR  = f"{INPUT_DIR}/selected-contexts"
MERGE_TARGET = 1800      # ký tự — kích thước đoạn thứ hai đem thử
OUTPUT_DIR = "/kaggle/working/outputs"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

assert os.path.isdir(CTX_DIR), f"KHÔNG thấy {CTX_DIR}"
assert os.path.isfile(SCORES), f"CHƯA UPLOAD {SCORES} — xem cell markdown ở trên"

## Bước 1 — Dựng lại mốc, rồi tìm 24 gold hụt

`base n=2` phải ra **0.8883**, `max n=2` phải ra **0.9083**. Lệch là thiết lập sai —
dừng, đừng đọc tiếp.

In [ ]:
dev  = json.load(open(DEV_GOLD, encoding="utf-8"))
S    = json.load(open(SCORES,   encoding="utf-8"))
gold = {q: {str(x) for x in v["answer"]} for q, v in dev.items()}
dev_q = {q: v["question"] for q, v in dev.items()}
bm25  = {q: sorted(S[q], key=lambda d: -S[q][d]["bm25"]) for q in dev_q}

def top5(q, scores=None, variant="max"):
    s = scores if scores is not None else S[q]
    return blend_bm25_first(DC.rank_by(s, variant), bm25[q], k=5, n_bm25=2)

def rec(variant):
    return sum(len(gold[q] & set(top5(q, variant=variant))) / len(gold[q])
               for q in gold) / len(gold)

for v, want in (("base", 0.8883), ("max", 0.9083)):
    got = rec(v)
    ok = "OK" if abs(got - want) < 0.002 else "<-- LỆCH, DỪNG LẠI"
    print(f"{v:5s} n=2 = {got:.4f}  (phải là {want})  {ok}")

MISS = [(q, d) for q in dev_q for d in gold[q] if d not in top5(q) and d in S[q]]
print(f"\n{len(MISS)} gold hụt top-5 nhưng còn trong top-100  (mong đợi 24)")

## Bước 2 — Băm ở hai kích thước, ĐẾM trước khi chấm

Quy tắc 6: ước chi phí bằng số đoạn. Mong đợi ~5.100 (hiện tại) + ~1.900 (gộp) ≈ 7.000
đoạn ≈ 9 phút ở nhịp 12,9 đoạn/s. Vượt xa là có gì sai, dừng lại.

In [ ]:
def merged(parts, target=MERGE_TARGET):
    """Gộp đoạn liền kề cho tới ~target ký tự. chunks_of cắt theo Điều nên đoạn
    trung bình chỉ 640 ký tự -> K=20 phủ ~16% văn bản. Gộp lên 1800 thì phủ ~43%."""
    out, buf = [], ""
    for p in parts:
        if buf and len(buf) + len(p) + 2 > target:
            out.append(buf)
            buf = p
        else:
            buf = f"{buf}\n\n{p}" if buf else p
    if buf:
        out.append(buf)
    return out

VARIANTS = ("cur", f"merge{MERGE_TARGET}")
CH = {}                                   # (variant, doc_id) -> list[str]
for _, d in MISS:
    if ("cur", d) in CH:
        continue
    cur = chunks_of(read_passage(CTX_DIR, d))
    CH[("cur", d)], CH[(f"merge{MERGE_TARGET}", d)] = cur, merged(cur)

# Đếm theo CẶP (q, d), KHÔNG theo văn bản: doc 102434 xuất hiện ở 2 câu hỏi khác nhau
# nên bị chấm 2 lần với 2 câu hỏi. Đếm theo văn bản là hụt đúng 744 đoạn.
n = {v: sum(len(CH[(v, d)]) for _, d in MISS) for v in VARIANTS}
tong = sum(n.values())
print(f"{len(MISS)} cặp (câu, văn bản) | {len(CH)//2} văn bản riêng biệt\n")
print(f"{'':12s} {'đoạn':>7s} {'dài TB':>7s} {'đoạn/vb':>8s} {'K=20 phủ':>9s}")
for v in VARIANTS:
    dai = sum(len(c) for _, d in MISS for c in CH[(v, d)]) / n[v]
    per = n[v] / len(MISS)
    print(f"{v:12s} {n[v]:7,} {dai:7.0f} {per:8.0f} {min(1, 20/per):9.0%}")
print(f"\nTỔNG {tong:,} đoạn ≈ {tong/12.9/60:.1f} phút ở nhịp 12,9 đoạn/s")
assert tong < 30_000, "quá nhiều đoạn — kiểm lại trước khi đốt GPU"

## Bước 3 — Load model + chấm TOÀN BỘ đoạn

Một lần `.predict()` cho tất cả — batch tí hon thì overhead dispatch GPU ăn hết thời gian
(bug hiệu năng đã ghi trong `rerank_from_d.py`).

In [ ]:
score_fn = load_reranker("AITeamVN/Vietnamese_Reranker", device="cuda")

pairs, owner = [], []
for q, d in MISS:
    for v in ("cur", f"merge{MERGE_TARGET}"):
        for i, c in enumerate(CH[(v, d)]):
            pairs.append([dev_q[q], c])
            owner.append((v, q, d, i))

t0 = time.time()
sc = score_fn.predict(pairs, show_progress_bar=True)
print(f"chấm {len(pairs):,} đoạn trong {(time.time()-t0)/60:.1f} phút"
      f"  ({len(pairs)/(time.time()-t0):.1f} đoạn/s)")

ORACLE = {}
for (v, q, d, i), s in zip(owner, sc):
    ORACLE.setdefault(v, {}).setdefault(q, {}).setdefault(d, []).append((i, float(s)))
for v in ORACLE:
    for q in ORACLE[v]:
        for d in ORACLE[v][q]:
            ORACLE[v][q][d] = [s for _, s in sorted(ORACLE[v][q][d])]   # theo thứ tự đoạn

## Bước 4 — LƯU NGAY (quy tắc 2)

Lưu trước khi phân tích. Đây là tài sản của lượt GPU này — phần dưới hỏng cũng không sao,
phân tích lại được trên CPU ở nhà.

Điểm lưu **theo đúng thứ tự đoạn** mà `chunks_of` sinh ra, nên ở nhà chỉ cần chạy lại
`chunks_of(read_passage(...))` là khớp chỉ số — không cần lưu text (tiết kiệm ~7MB).

In [ ]:
p = f"{OUTPUT_DIR}/oracle_chunks_dev300.json"
json.dump({"scores": ORACLE, "miss": MISS, "merge_target": MERGE_TARGET,
           "model": "AITeamVN/Vietnamese_Reranker", "n_pairs": len(pairs)},
          open(p, "w", encoding="utf-8"), ensure_ascii=False)
print(f"ĐÃ LƯU {p}  ({os.path.getsize(p):,} bytes) — TẢI VỀ dù phần dưới có hỏng")

## Bước 5 — TRẦN: chọn đoạn hoàn hảo thì cứu được mấy câu?

Thay `ce_deep` của riêng văn bản gold bằng **max trên TOÀN BỘ đoạn**, giữ nguyên điểm
của mọi văn bản đối thủ, rồi chốt lại top-5. Đây là chặn trên: không hàm chọn đoạn nào
vượt được "chọn hoàn hảo".

In kèm `K=20 hiện tại` để thấy khoảng cách — nếu hai cột bằng nhau thì hàm chọn đoạn
hiện tại **đã** tìm ra đoạn tốt nhất, và vấn đề nằm ở chỗ khác hoàn toàn.

In [ ]:
def oracle(variant):
    """Nâng ce_deep của MỌI văn bản gold hụt lên max toàn bộ đoạn, cùng lúc.
    Đối thủ giữ nguyên điểm K=20 -> đây là chặn TRÊN của mọi hàm chọn đoạn."""
    S2 = {q: {d: dict(v) for d, v in S[q].items()} for q in dev_q}
    for q, d in MISS:
        S2[q][d]["ce_deep"] = max(ORACLE[variant][q][d])
    got = [(q, d) for q, d in MISS if d in top5(q, S2[q])]
    r = sum(len(gold[q] & set(top5(q, S2[q]))) / len(gold[q]) for q in gold) / len(gold)
    return got, r

print(f"{'kích thước đoạn':16s} {'cứu':>6s} {'recall@5':>10s} {'Δ vs 0.9083':>12s}")
res = {}
for v in VARIANTS:
    g, r = oracle(v)
    res[v] = g
    print(f"{v:16s} {len(g):3d}/{len(MISS):<2d} {r:10.4f} {(r-0.9083)*100:+11.2f}")

best = max(VARIANTS, key=lambda v: len(res[v]))
n_best = len(res[best])
print("\n" + "=" * 64)
if n_best >= 6:
    print(f"ĐI TIẾP: chọn đoạn là nút thắt ({best}, cứu {n_best} câu).")
    print("Viết pick_chunks bản embedding, đo lại trên dev300.")
elif n_best <= 2:
    print(f"DỪNG: chỉ cứu được {n_best} câu kể cả khi chọn đoạn HOÀN HẢO.")
    print("KHÔNG phải vấn đề chọn đoạn — đừng đốt 2,5h GPU cho embedding.")
else:
    print(f"VÙNG XÁM: cứu {n_best} câu. Trần đã sát ngưỡng 2,0 của quy tắc 4,")
    print("mà thực tế chỉ ăn được một phần trần. Cân nhắc kỹ trước khi đầu tư.")
print("=" * 64)

# K=20 hiện tại đã tìm ra đoạn tốt nhất chưa? Chỉ xét văn bản ĐÃ được chấm sâu.
da_sau = [(q, d) for q, d in MISS if "ce_deep" in S[q][d]]
trung = sum(max(ORACLE["cur"][q][d]) <= S[q][d]["ce_deep"] + 1e-6 for q, d in da_sau)
print(f"\ntrong {len(da_sau)} văn bản ĐÃ chấm sâu: {trung} cái K=20 vốn đã tìm đúng đoạn tốt nhất")
print(f"                                    {len(da_sau)-trung} cái bỏ sót đoạn tốt hơn  <- phần embedding cứu được")

for v in VARIANTS:
    print(f"\n{v} cứu: " + ", ".join(f"{q}/{d}" for q, d in res[v]))

## Bước 6 — Đoạn thắng nằm ở hạng mấy theo tiêu chí TỪ TRÙNG hiện tại?

Đây là chỗ chỉ thẳng ra cần sửa gì. Với mỗi câu cứu được: đoạn điểm cao nhất đang bị
`pick_chunks` xếp hạng bao nhiêu?

- hạng **> 20** → K=20 chưa từng nhìn thấy nó → **đổi tiêu chí chọn** (embedding) là đúng
- hạng **≤ 20** → nó ĐÃ được chấm rồi mà vẫn thua → đổi tiêu chí chọn **vô ích**, vấn đề
  nằm ở cách gộp điểm hoặc ở chính cross-encoder

In [ ]:
from make_candidates_fallback import tok

print(f"{'qid':>8s} {'doc':>8s} {'#đoạn':>6s} {'hạng đoạn thắng':>16s}   {'K=20 thấy?':>10s}")
ngoai = 0
for q, d in res["cur"]:
    parts = CH[("cur", d)]
    qs = set(tok(dev_q[q]))
    order = sorted(range(len(parts)), key=lambda i: -len(qs & set(tok(parts[i]))))
    win = max(range(len(parts)), key=lambda i: ORACLE["cur"][q][d][i])
    r = order.index(win) + 1
    ngoai += r > 20
    print(f"{q:>8s} {d:>8s} {len(parts):>6d} {r:>16d}   {'KHÔNG' if r > 20 else 'có':>10s}")

print(f"\n{ngoai}/{len(res['cur'])} câu có đoạn thắng nằm NGOÀI top-20 của tiêu chí từ trùng")
print("-> đó chính là phần embedding có cửa cứu. Phần còn lại thì không.")

for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"\n  {f}  {os.path.getsize(os.path.join(OUTPUT_DIR, f)):,} bytes")
print("TẢI outputs/ VỀ TRƯỚC KHI ĐÓNG PHIÊN")